# Tagged units vs. SI/MA population waveforms

Using the two artifacts already saved to disk:

1. the **big waveform/feature CSV** (`all_sessions_waveform_features.csv`, built
   by `waveform_features_dataset.ipynb`), and
2. the **opto-tagging metric CSVs** (`*_laser_response_metrics.csv`, from
   `optotagging_Anna_nwb_batch.ipynb`),

this notebook selects the **SI + MA** (ventral pallidum) units and the
**opto-tagged** units, then overlays their waveforms and features in the same
figures so you can see whether the tagged units stand out from the SI/MA
population.

No NWB re-reading is needed — everything comes from the two CSV sources.


## 1. Imports & configuration

In [ ]:
# =============================================================================
# ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
import re
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from waveform_clustering import (
    FEATURE_COLS,
    load_features_dataset,
    normalize_waveform,
)

%matplotlib inline

# ---- Configuration -------------------------------------------------------
# Big per-unit dataset (all sessions, all units) from waveform_features_dataset.ipynb
DATASET_CSV = Path("/root/capsule/scratch/waveform_clustering/all_sessions_waveform_features.csv")

# Folder holding the opto-tagging metric CSVs (*_laser_response_metrics.csv)
OPTO_DIR = Path("/root/capsule/scratch/opto_tagging_Anna")

# Ventral pallidum is covered by the CCF acronyms SI + MA.
TARGET_REGIONS = ["SI", "MA"]

# Opto-tagging thresholds (same as the batch notebook).
RED_MIN_SIG_PULSES = 4     # red: >= 4 significant pulses
BLUE_MIN_SIG_PULSES = 5    # blue: >= 5 significant pulses
MAX_JITTER = 0.01          # s
MAX_ISI = 0.5              # pre-stim ISI-violation ratio

feature_cols = FEATURE_COLS

## 2. Load the big waveform / feature dataset

Reload the flat CSV back into a feature/metadata table plus a waveform matrix,
then keep only the **SI / MA** units.


In [ ]:
loaded = load_features_dataset(DATASET_CSV)
features = loaded["features"]
waveforms = loaded["waveforms"]
time_ms = loaded["time_ms"]

# Amplitude-normalize every stored (baseline-corrected, trough-aligned) waveform.
norm_waveforms = (
    np.vstack([normalize_waveform(w) for w in waveforms]) if len(waveforms) else waveforms
)

region_mask = features["region"].isin(TARGET_REGIONS).values
print(f"Total units in dataset: {len(features)}")
print(f"SI/MA units: {int(region_mask.sum())}")
print(features.loc[region_mask, "region"].value_counts())

## 3. Identify the opto-tagged units

Read each `*_laser_response_metrics.csv`, apply the same tagging query used by
the batch notebook (red: >= `RED_MIN_SIG_PULSES` significant pulses; blue: >=
`BLUE_MIN_SIG_PULSES`, excluding units already red-tagged), and collect the
`(session, unit_index)` pairs. The session core (e.g. `839480_2026-06-03_...`)
is parsed from the filename so it matches the dataset's `session_name`.


In [ ]:
SESSION_CORE_RE = re.compile(r"(\d+_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")


def tagged_from_metrics(metrics, trial_type, min_sig_pulses, max_jitter, max_isi):
    """Rows of `metrics` passing the tagging criteria for one trial type."""
    q = []
    col_pulses = f"{trial_type}_train_max_num_sig_pulses"
    col_jitter = f"{trial_type}_train_best_mean_jitter"
    if col_pulses in metrics.columns:
        q.append(f"{col_pulses} >= {min_sig_pulses}")
    if col_jitter in metrics.columns:
        q.append(f"{col_jitter} < {max_jitter}")
    if "pre_stim_isi_ratio" in metrics.columns:
        q.append(f"pre_stim_isi_ratio < {max_isi}")
    if not q:
        return metrics.iloc[0:0]
    return metrics.query(" and ".join(q))


def load_tagged_units(opto_dir, red_min_sig_pulses, blue_min_sig_pulses,
                      max_jitter, max_isi):
    """Collect tagged (session_name, unit_index, tag_type) from metric CSVs."""
    rows = []
    csvs = sorted(Path(opto_dir).glob("*_laser_response_metrics.csv"))
    print(f"Found {len(csvs)} metric CSV(s) in {opto_dir}")
    for csv in csvs:
        m = SESSION_CORE_RE.search(csv.name)
        if not m:
            print(f"  [skip] cannot parse session from {csv.name}")
            continue
        session_core = m.group(1)
        metrics = pd.read_csv(csv)
        if "unit_id" not in metrics.columns:
            continue
        trial_types = sorted({
            c[: -len("_train_max_num_sig_pulses")]
            for c in metrics.columns if c.endswith("_train_max_num_sig_pulses")
        })
        red_types = [t for t in trial_types if "red" in t]
        blue_types = [t for t in trial_types if "blue" in t]

        red_idx = set()
        for tt in red_types:
            tg = tagged_from_metrics(metrics, tt, red_min_sig_pulses, max_jitter, max_isi)
            red_idx.update(tg.index.tolist())
            for uid in tg["unit_id"].astype(int):
                rows.append((session_core, int(uid), tt))
        for tt in blue_types:
            tg = tagged_from_metrics(metrics, tt, blue_min_sig_pulses, max_jitter, max_isi)
            tg = tg[~tg.index.isin(red_idx)]
            for uid in tg["unit_id"].astype(int):
                rows.append((session_core, int(uid), tt))

    tagged = pd.DataFrame(rows, columns=["session_name", "unit_index", "tag_type"])
    return tagged.drop_duplicates(subset=["session_name", "unit_index"]).reset_index(drop=True)


tagged_df = load_tagged_units(
    OPTO_DIR,
    red_min_sig_pulses=RED_MIN_SIG_PULSES,
    blue_min_sig_pulses=BLUE_MIN_SIG_PULSES,
    max_jitter=MAX_JITTER,
    max_isi=MAX_ISI,
)
print(f"\nTagged units found: {len(tagged_df)}")
print(tagged_df["tag_type"].value_counts())
tagged_df.head()

## 4. Match tagged units into the dataset

Join the tagged `(session, unit)` pairs onto the big dataset so each tagged unit
picks up its stored waveform + features. Then build masks for the SI/MA
population and the tagged units.


In [ ]:
# Build a matching key on both sides.
def make_key(df):
    return (df["session_name"].astype(str) + "|" + df["unit_index"].astype(int).astype(str))

feat_key = make_key(features)
tagged_key = set(make_key(tagged_df))

tagged_mask = feat_key.isin(tagged_key).values

n_tagged_matched = int(tagged_mask.sum())
n_tagged_in_sima = int((tagged_mask & region_mask).sum())
print(f"Tagged units matched in dataset: {n_tagged_matched} / {len(tagged_df)}")
print(f"  of those in SI/MA: {n_tagged_in_sima}")
print(f"SI/MA population units: {int(region_mask.sum())}")

if n_tagged_matched < len(tagged_df):
    missing = len(tagged_df) - n_tagged_matched
    print(f"\n[note] {missing} tagged unit(s) were not found in the dataset "
          "(their opto session may not be in the dataset, or the unit produced "
          "no valid waveform).")

# Region breakdown of the tagged units that matched.
print("\nTagged units by region:")
print(features.loc[tagged_mask, "region"].fillna("None").value_counts())

## 5. Overlay waveforms: SI/MA population vs. tagged units

Grey = SI/MA population; black = its mean. Coloured = tagged units; bold =
their mean. This is the direct visual test of whether the tagged units differ.


In [ ]:
pop_mask = region_mask                    # SI/MA population
tag_mask = tagged_mask                     # all matched tagged units

fig, ax = plt.subplots(figsize=(9, 6))

# SI/MA population
pop_wf = norm_waveforms[pop_mask]
for w in pop_wf:
    ax.plot(time_ms, w, color="lightgrey", linewidth=0.4, zorder=1)
if len(pop_wf):
    ax.plot(time_ms, pop_wf.mean(axis=0), color="black", linewidth=2.5,
            label=f"SI/MA mean (n={len(pop_wf)})", zorder=3)

# Tagged units
tag_wf = norm_waveforms[tag_mask]
for w in tag_wf:
    ax.plot(time_ms, w, color="tab:red", linewidth=0.7, alpha=0.5, zorder=2)
if len(tag_wf):
    ax.plot(time_ms, tag_wf.mean(axis=0), color="tab:red", linewidth=2.5,
            label=f"Tagged mean (n={len(tag_wf)})", zorder=4)

ax.set_xlabel("Time (ms)")
ax.set_ylabel("Normalized amplitude")
ax.set_title("SI/MA population vs. opto-tagged units")
ax.legend()
plt.tight_layout()
plt.show()

### 5a. Compare only tagged units that lie in SI/MA

Restrict the tagged group to SI/MA so both groups are anatomically matched.


In [ ]:
tag_in_sima = tagged_mask & region_mask
pop_only = region_mask & ~tagged_mask     # SI/MA, untagged

fig, ax = plt.subplots(figsize=(9, 6))
pop_wf = norm_waveforms[pop_only]
for w in pop_wf:
    ax.plot(time_ms, w, color="lightgrey", linewidth=0.4, zorder=1)
if len(pop_wf):
    ax.plot(time_ms, pop_wf.mean(axis=0), color="black", linewidth=2.5,
            label=f"SI/MA untagged mean (n={len(pop_wf)})", zorder=3)

tag_wf = norm_waveforms[tag_in_sima]
for w in tag_wf:
    ax.plot(time_ms, w, color="tab:red", linewidth=1.0, alpha=0.7, zorder=2)
if len(tag_wf):
    ax.plot(time_ms, tag_wf.mean(axis=0), color="tab:red", linewidth=2.5,
            label=f"SI/MA tagged mean (n={len(tag_wf)})", zorder=4)

ax.set_xlabel("Time (ms)")
ax.set_ylabel("Normalized amplitude")
ax.set_title("SI/MA: untagged vs. tagged units")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Compare waveform features

Overlay the two groups in feature space and compare the key separators
(trough-to-peak duration, half-width).


In [ ]:
# Feature scatter: SI/MA population vs tagged units
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.scatter(features.loc[pop_mask, "trough_to_peak_ms"],
           features.loc[pop_mask, "half_width_ms"],
           s=25, c="lightgrey", edgecolor="grey", linewidth=0.3,
           label=f"SI/MA (n={int(pop_mask.sum())})", zorder=1)
ax.scatter(features.loc[tag_mask, "trough_to_peak_ms"],
           features.loc[tag_mask, "half_width_ms"],
           s=90, c="tab:red", marker="*", edgecolor="black", linewidth=0.5,
           label=f"Tagged (n={int(tag_mask.sum())})", zorder=2)
ax.set_xlabel("Trough-to-peak duration (ms)")
ax.set_ylabel("Half-width (ms)")
ax.set_title("Waveform feature space: SI/MA vs. tagged")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distributions of key features (SI/MA population vs tagged)
compare_cols = ["trough_to_peak_ms", "half_width_ms", "peak_trough_ratio", "repolarization_slope"]
fig, axes = plt.subplots(1, len(compare_cols), figsize=(4 * len(compare_cols), 4))
for ax, col in zip(axes, compare_cols):
    ax.hist(features.loc[pop_mask, col], bins=25, color="lightgrey",
            alpha=0.9, density=True, label="SI/MA")
    tvals = features.loc[tag_mask, col].dropna()
    for v in tvals:
        ax.axvline(v, color="tab:red", alpha=0.5, linewidth=1)
    if len(tvals):
        ax.axvline(tvals.mean(), color="tab:red", linewidth=2.5, label="tagged mean")
    ax.set_title(col)
    ax.set_xlabel(col)
axes[0].set_ylabel("Density")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Numeric summary: mean +/- std of each feature for the two groups.
summary = pd.DataFrame({
    "SI/MA_mean": features.loc[pop_mask, feature_cols].mean(),
    "SI/MA_std": features.loc[pop_mask, feature_cols].std(),
    "tagged_mean": features.loc[tag_mask, feature_cols].mean(),
    "tagged_std": features.loc[tag_mask, feature_cols].std(),
})
summary["n_SI/MA"] = int(pop_mask.sum())
summary["n_tagged"] = int(tag_mask.sum())
summary

## 7. Notes

- Grey = SI/MA population (from the big dataset CSV); red = opto-tagged units
  (matched from the `*_laser_response_metrics.csv` files by session + unit).
- Section 5 overlays tagged units on the whole SI/MA population; section 5a
  restricts the comparison to tagged units that are themselves in SI/MA.
- Adjust `RED_MIN_SIG_PULSES` / `BLUE_MIN_SIG_PULSES` / `MAX_JITTER` /
  `MAX_ISI` in the config cell to match the thresholds you used when tagging.
- If few/no tagged units match, check that the opto sessions are included in the
  big dataset and that `OPTO_DIR` points at the saved metric CSVs.
